# Testing Image Editing Tools

This notebook exercises each registered image-editing tool using `sample1.jpg`. It makes real model calls and may download weights when they are not cached.

In [ ]:
from pathlib import Path

from framework.utils.project import find_project_root
WORKSPACE = find_project_root()

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scipy.ndimage import distance_transform_edt
from framework.inference import ModelCache
from framework.tools import image_editing_tools

models = ModelCache()
tools = image_editing_tools(model_cache=models)
tool_dict = {tool.name: tool for tool in tools}
expected_tools = {
    'apply_filter', 'edit_object', 'flip_object', 'generate_object',
    'move_object', 'remove_object', 'replace_object', 'rotate_object',
    'scale_object', 'segment_object', 'shift_object',
}
assert set(tool_dict) == expected_tools, sorted(tool_dict)
print('Workspace:', WORKSPACE)
print('Current tools:', ', '.join(sorted(tool_dict)))

In [ ]:
img = Image.open(WORKSPACE / 'resources' / 'test_images' / 'sample1.jpg').convert('RGB')
mask = tool_dict['segment_object'](image=img, prompt='cat')
mask = mask.convert('L').resize(img.size, Image.Resampling.NEAREST)
mask_array = np.asarray(mask) > 0
assert mask_array.any(), 'Text segmentation returned an empty cat mask.'
cat_box = list(mask.getbbox())
distance = distance_transform_edt(mask_array)
point_y, point_x = np.unravel_index(np.argmax(distance), distance.shape)
point = [int(point_x), int(point_y)]
row_xs = np.flatnonzero(mask_array[point_y])
runs = np.split(row_xs, np.where(np.diff(row_xs) > 1)[0] + 1)
point_run = next(run for run in runs if point_x in run)
scribble_xs = np.linspace(point_run[0], point_run[-1], min(7, len(point_run)), dtype=int)
scribble = [[int(x), int(point_y)] for x in scribble_xs]
selection_mask = mask
selection_box = cat_box
print('Image size:', img.size)
print('Cat box from SAM3 mask:', cat_box)
print('Interior positive point:', point)
print('Positive scribble points:', scribble)
plt.imshow(img)
plt.imshow(mask, alpha=0.45, cmap='Reds')
plt.scatter([point[0]], [point[1]], c='cyan', s=30)
plt.plot([p[0] for p in scribble], [p[1] for p in scribble], c='yellow', linewidth=2)
plt.axis('off')
plt.show()

In [ ]:
def assert_image(result, label):
    assert isinstance(result, Image.Image), f'{label} did not return a Pillow image'
    assert result.size == img.size, f'{label} changed the image dimensions'
    return result

def expect_error(error_type, message, call):
    try:
        call()
    except error_type as error:
        assert message in str(error), f'Unexpected error: {error}'
    else:
        raise AssertionError(f'Expected {error_type.__name__}: {message}')

filter_names = ('blackWhite', 'invert', 'blur', 'brightness', 'contrast', 'saturation')
global_filters = {
    name: assert_image(tool_dict['apply_filter'](image=img, filter_name=name, value=0.15), f'global {name}')
    for name in filter_names
}
object_filters = {
    name: assert_image(tool_dict['apply_filter'](image=img, filter_name=name, value=0.15, mask=mask), f'object {name}')
    for name in filter_names
}
outside_y, outside_x = np.argwhere(~mask_array)[0]
assert object_filters['invert'].getpixel((int(outside_x), int(outside_y))) == img.getpixel((int(outside_x), int(outside_y)))
assert_image(tool_dict['apply_filter'](image=img, filter_name='brightness', value=-1), 'minimum brightness')
assert_image(tool_dict['apply_filter'](image=img, filter_name='contrast', value=1), 'maximum contrast')
expect_error(ValueError, 'unsupported filter', lambda: tool_dict['apply_filter'](image=img, filter_name='not-a-filter', value=0))
expect_error(ValueError, 'between -1 and 1', lambda: tool_dict['apply_filter'](image=img, filter_name='blur', value=1.01))
expect_error(ValueError, 'mask is empty', lambda: tool_dict['apply_filter'](image=img, filter_name='invert', value=0, mask=Image.new('L', img.size, 0)))
print('apply_filter passed: six filters globally and masked, boundary preservation, strength endpoints, and invalid inputs.')
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, name in zip(axes.flat, filter_names):
    axis.imshow(global_filters[name]); axis.set_title(name); axis.axis('off')
plt.show()

In [ ]:
assert selection_mask.size == img.size
assert selection_mask.getbbox() == tuple(cat_box)
assert len(point) == 2 and len(scribble) >= 2
assert all(selection_mask.getpixel((x, y)) > 0 for x, y in [point, *scribble])
print('All geometric inputs come from the real SAM3 cat mask:', cat_box)

## Model-backed examples

The setup cell uses the current workspace source and one shared lazy model cache. The sample cat is segmented once by text; the point, box, and scribble inputs below are derived from that returned mask. Model calls may be slow on CPU/MPS and may download weights if they are not cached.

In [ ]:
segment_cases = {
    'text prompt': tool_dict['segment_object'](image=img, prompt='cat'),
    'point': tool_dict['segment_object'](image=img, point=point),
    'box from cat mask': tool_dict['segment_object'](image=img, box=cat_box),
    'scribble on cat': tool_dict['segment_object'](image=img, scribble=scribble),
}
for name, result in segment_cases.items():
    assert isinstance(result, Image.Image) and result.mode == 'L'
    assert result.size == img.size and result.getbbox() is not None, f'{name} returned an invalid mask'
print('segment_object passed all four selector paths; selected-pixel counts:', {name: int(np.count_nonzero(result)) for name, result in segment_cases.items()})
expect_error(ValueError, 'exactly one', lambda: tool_dict['segment_object'](image=img))
expect_error(ValueError, 'exactly one', lambda: tool_dict['segment_object'](image=img, prompt='cat', point=point))
expect_error(ValueError, 'non-empty string', lambda: tool_dict['segment_object'](image=img, prompt='  '))
expect_error(ValueError, 'scribble must contain', lambda: tool_dict['segment_object'](image=img, scribble=[]))
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for axis, (name, result) in zip(axes, segment_cases.items()):
    axis.imshow(img); axis.imshow(result, alpha=0.45, cmap='Reds'); axis.set_title(name); axis.axis('off')
plt.show()

In [ ]:
removed = assert_image(tool_dict['remove_object'](image=img, mask=mask), 'remove_object')
edit_cases = {
    'local': tool_dict['edit_object'](image=img, mask=mask, prompt='make the cat fur black and white', mode='local'),
    'regional default padding': tool_dict['edit_object'](image=img, mask=mask, prompt='add a small blue hat to the cat', mode='regional'),
    'regional zero padding': tool_dict['edit_object'](image=img, mask=mask, prompt='change the cat collar to green', mode='regional', padding=0),
    'regional max padding': tool_dict['edit_object'](image=img, mask=mask, prompt='add a small bow beside the cat', mode='regional', padding=1),
    'global': tool_dict['edit_object'](image=img, mask=mask, prompt='make the room look like a sunny garden', mode='global'),
}
for name, result in edit_cases.items():
    assert_image(result, f'edit_object {name}')
print('remove_object and edit_object passed: local, regional padding 0/default/1, and global modes.')
expect_error(ValueError, "mode must be", lambda: tool_dict['edit_object'](image=img, mask=mask, prompt='edit', mode='unknown'))
expect_error(ValueError, 'padding must be between 0 and 1', lambda: tool_dict['edit_object'](image=img, mask=mask, prompt='edit', mode='regional', padding=1.1))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes.flat[0].imshow(removed); axes.flat[0].set_title('remove'); axes.flat[0].axis('off')
for axis, (name, result) in zip(axes.flat[1:], edit_cases.items()):
    axis.imshow(result); axis.set_title(name); axis.axis('off')
plt.show()

In [ ]:
import gc, torch
models.close(); gc.collect()
if torch.backends.mps.is_available(): torch.mps.empty_cache()
elif torch.cuda.is_available(): torch.cuda.empty_cache()
filled = assert_image(tool_dict['generate_object'](image=img, mask=mask, prompt='add a small red toy ball on the floor'), 'masked Klein generation')
boxed = assert_image(tool_dict['generate_object'](image=img, prompt='add a small red toy ball on the floor', box=cat_box), 'box Klein generation')
models.close(); gc.collect()
if torch.backends.mps.is_available(): torch.mps.empty_cache()
elif torch.cuda.is_available(): torch.cuda.empty_cache()
phrase_guided = assert_image(tool_dict['generate_object'](image=img, prompt='add the requested object naturally', box=cat_box, phrase='a red toy ball'), 'phrase-guided Klein generation')
print('generate_object passed: mask, box, and phrase-guided Klein regional edits.')
expect_error(ValueError, 'non-empty string', lambda: tool_dict['generate_object'](image=img, prompt=' ', mask=mask))
expect_error(ValueError, 'Provide one mask or one box', lambda: tool_dict['generate_object'](image=img, prompt='fill this'))
expect_error(ValueError, 'Provide a mask or a box, not both', lambda: tool_dict['generate_object'](image=img, prompt='fill this', mask=mask, box=cat_box))
expect_error(ValueError, 'phrase requires a box', lambda: tool_dict['generate_object'](image=img, prompt='photo', phrase='a toy ball'))
expect_error(ValueError, 'not a mask', lambda: tool_dict['generate_object'](image=img, prompt='photo', mask=mask, box=cat_box, phrase='a toy ball'))
expect_error(ValueError, 'phrase must be a non-empty string', lambda: tool_dict['generate_object'](image=img, prompt='photo', box=cat_box, phrase=' '))
expect_error(ValueError, 'positive width and height', lambda: tool_dict['generate_object'](image=img, prompt='photo', box=[cat_box[2], cat_box[3], cat_box[0], cat_box[1]]))
for title, output in [('masked Klein edit', filled), ('box Klein edit', boxed), ('phrase-guided Klein edit', phrase_guided)]:
    plt.figure(figsize=(5, 6)); plt.imshow(output); plt.title(title); plt.axis('off'); plt.show()

In [ ]:
import gc, torch
models.close(); gc.collect()
if torch.backends.mps.is_available(): torch.mps.empty_cache()
elif torch.cuda.is_available(): torch.cuda.empty_cache()
replacement = Image.new('RGBA', (256, 256), (0, 0, 0, 0))
ImageDraw.Draw(replacement).ellipse((12, 12, 244, 244), fill=(220, 35, 35, 255))
target_box = [0, 0, round(img.width * 0.42), round(img.height * 0.48)]
transform_cases = {
    'replace with transparent red disk': tool_dict['replace_object'](image=img, mask=mask, replacement=replacement),
    'move to center point': tool_dict['move_object'](image=img, mask=mask, target=[img.width * 0.72, img.height * 0.5]),
    'move and fit into box': tool_dict['move_object'](image=img, mask=mask, target=target_box),
    'scale down': tool_dict['scale_object'](image=img, mask=mask, factor=0.75),
    'scale up': tool_dict['scale_object'](image=img, mask=mask, factor=1.25),
    'flip horizontally': tool_dict['flip_object'](image=img, mask=mask, axis='x'),
    'flip vertically': tool_dict['flip_object'](image=img, mask=mask, axis='y'),
    'rotate counterclockwise': tool_dict['rotate_object'](image=img, mask=mask, angle=20),
    'rotate clockwise': tool_dict['rotate_object'](image=img, mask=mask, angle=-20),
    'shift right': tool_dict['shift_object'](image=img, mask=mask, offset=[40, 0]),
    'shift left': tool_dict['shift_object'](image=img, mask=mask, offset=[-40, 0]),
}
for name, result in transform_cases.items():
    assert_image(result, name)
print('Object transforms passed:', ', '.join(transform_cases))
expect_error(ValueError, "axis must be", lambda: tool_dict['flip_object'](image=img, mask=mask, axis='z'))
expect_error(ValueError, 'greater than zero', lambda: tool_dict['scale_object'](image=img, mask=mask, factor=0))
expect_error(ValueError, 'each target must be', lambda: tool_dict['move_object'](image=img, mask=mask, target=[1, 2, 3]))
expect_error(ValueError, 'offset must be', lambda: tool_dict['shift_object'](image=img, mask=mask, offset=[1]))
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for axis, (name, result) in zip(axes.flat, transform_cases.items()):
    axis.imshow(result); axis.set_title(name); axis.axis('off')
for axis in axes.flat[len(transform_cases):]: axis.axis('off')
plt.tight_layout(); plt.show()